In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path

import re
import unicodedata

In [ ]:
municipalities = gpd.read_file(snakemake.input.municipalities_path)

In [ ]:
municipalities.sort_values(by='nummer')

In [ ]:
municipality_names_map = (
    municipalities
    .drop_duplicates(subset="nummer")
    .set_index("nummer")["navn"]
)

In [ ]:
year = 2025

In [ ]:
data_url = f"https://data.elhub.no/download/consumption_per_group_municipality_hour/2025/consumption_per_group_municipality_hour-all-no-{year}-00-00.csv"

In [ ]:
df = pd.read_csv(data_url,sep=';',decimal=",")
df["datetime"] = pd.to_datetime(df["SLUTTID"], utc=True, errors="coerce")

In [ ]:
# https://elhub.atlassian.net/wiki/spaces/Data/pages/1171587077/Forbruksgrupper

In [ ]:
demand = (
    df
    .copy()
    .set_index('datetime')
    .assign(demand_MW = lambda x : x.VOLUM_KWH/1000)
    .loc[:,['KOMMUNE','KOMMUNENUMMER','FORBRUKSGRUPPE','demand_MW']]
    .replace({'Privat' : 'Existing residential', 'Industri' : 'Existing industry', 'Forretning' : 'Existing commercial'})
    .query("KOMMUNENUMMER != 0")
)

demand

In [ ]:
supposed_demand = 139000000
loaded_demand = demand.demand_MW.sum()
missing_demand = supposed_demand - loaded_demand

In [ ]:
share_missing = supposed_demand/loaded_demand

In [ ]:
demand = (
    demand
    .assign(demand_MW = lambda x : x.demand_MW.mul(share_missing))
)

In [ ]:
# Hydrogen: 27 (+27) TWh
# Data centres: 19 (+18)  TWh
# Industry: 65 (+20) TWh
# Petroleum: 12 (+2) TWh
# Transport: 26 (+22) TWh
# General: 72 (-6) TWh

## Petroleum

In [ ]:
petro_shares = pd.read_csv(snakemake.input.petro_shares)

In [ ]:
petroleum_elec = petro_shares.set_index('municipality')['share_of_demand'].to_dict()

In [ ]:
## these are from demand_projections_NO.ods 
#petroleum_elec = {
#    'Tysvær' : 0.224,
#    'Samnanger' : 0.079,
#    'Farsund' : 0.012,
#    'Øygarden' : 0.193,
#    'Hammerfest' : 0.263,
#    'Aukra'  : 0.018,
#    'Hitra' : 0.042,
#    'Aure'  : 0.145,
#    'Alver' : 0.024,
#    }
#

In [ ]:
annual_petroleum_twh = 2
hours_in_year = 8760
petroleum_mw = {
    kommune: share * annual_petroleum_twh * 1_000_000 / hours_in_year
    for kommune, share in petroleum_elec.items()
}

In [ ]:
times = demand.index.unique().sort_values()

# Find the municipality number from the existing data
kommune_nummer = (
    demand[['KOMMUNE', 'KOMMUNENUMMER']]
    .drop_duplicates()
    .set_index('KOMMUNE')['KOMMUNENUMMER']
    .to_dict()
)

# Make one constant-demand row per hour and relevant municipality
petroleum_rows = []

for kommune, mw in petroleum_mw.items():
    for time in times:
        petroleum_rows.append({
            'datetime': time,
            'KOMMUNE': kommune,
            'KOMMUNENUMMER': kommune_nummer[kommune],
            'FORBRUKSGRUPPE': 'Petroleum electrification',
            'demand_MW': mw,
        })

df_petroleum = (
    pd.DataFrame(petroleum_rows)
    .set_index('datetime')
)

# Append petroleum demand to the original hourly demand data
demand_with_petroleum = pd.concat([demand, df_petroleum]).sort_index()

In [ ]:
daily_demand_by_group = (
    demand_with_petroleum
    .reset_index()
    .assign(demand_GW = lambda x : x.demand_MW/1000)
    .groupby(['datetime', 'FORBRUKSGRUPPE'])['demand_GW'].sum()
    .unstack('FORBRUKSGRUPPE')
    .resample('D').sum()
)

ax = daily_demand_by_group.plot(
    figsize=(16, 7),
    linewidth=1.2
)

ax.set_title('Daily average electricity demand by consumption group — Norway, 2050')
ax.set_xlabel('')
ax.set_ylabel('Daily demand [GWh]')
ax.legend(title='Consumption group', ncol=4)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## General

In [ ]:
# https://snl.no/alminnelig_forbruk
general_increase_twh = -6

In [ ]:
base = demand_with_petroleum.reset_index()

ordinary_groups = ['Existing residential', 'Existing commercial']

# Annual baseline energy by municipality.
# MW values at hourly resolution sum to MWh.
municipality_ordinary_mwh = (
    base.loc[base['FORBRUKSGRUPPE'].isin(ordinary_groups)]
    .groupby('KOMMUNENUMMER')['demand_MW']
    .sum()
)

# Municipality shares of national ordinary consumption
municipality_shares = (
    municipality_ordinary_mwh
    / municipality_ordinary_mwh.sum()
)

municipality_added_mwh = (
    municipality_shares
    * general_increase_twh
    * 1_000_000  # TWh -> MWh
)

In [ ]:
ordinary = (
    base.loc[base['FORBRUKSGRUPPE'].isin(['Existing residential', 'Existing commercial'])]
    .groupby(['datetime', 'KOMMUNE', 'KOMMUNENUMMER'], as_index=False)['demand_MW']
    .sum()
)

In [ ]:
# Total annual baseline MWh for each municipality
ordinary['annual_baseline_mwh'] = (
    ordinary.groupby('KOMMUNENUMMER')['demand_MW']
    .transform('sum')
)

In [ ]:
# Map annual added energy to each municipality
ordinary['added_annual_mwh'] = ordinary['KOMMUNENUMMER'].map(
    municipality_added_mwh
)

In [ ]:
# The hourly increment preserves the municipality's load shape.
# Sum of demand_MW across hours = MWh, since each observation is one hour.
ordinary['demand_MW'] = (
    ordinary['demand_MW']
    / ordinary['annual_baseline_mwh']
    * ordinary['added_annual_mwh']
)

df_general_profiled = ordinary.assign(
    FORBRUKSGRUPPE='General change'
)[
    [
        'datetime',
        'KOMMUNE',
        'KOMMUNENUMMER',
        'FORBRUKSGRUPPE',
        'demand_MW',
    ]
]

In [ ]:
demand_with_all_growth = pd.concat(
    [base, df_general_profiled],
    ignore_index=True,
).set_index('datetime')

In [ ]:
daily_demand_by_group = (
    demand_with_all_growth
    .reset_index()
    .assign(demand_GW = lambda x : x.demand_MW/1000)
    .groupby(['datetime', 'FORBRUKSGRUPPE'])['demand_GW'].sum()
    .unstack('FORBRUKSGRUPPE')
    .resample('D').sum()
)

ax = daily_demand_by_group.plot(
    figsize=(16, 7),
    linewidth=1.2
)

ax.set_title('Daily average electricity demand by consumption group — Norway, 2050')
ax.set_xlabel('')
ax.set_ylabel('Daily demand [GWh]')
ax.legend(title='Consumption group', ncol=4)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## New industry

In [ ]:
national_new_industry_demand_twh = 20

In [ ]:
# https://www.norskeutslipp.no/no/Komponenter/Utslipp/Klimagasser-CO2-ekvivalenter/?ComponentType=utslipp&ComponentPageID=1166&SectorID=600
industry_emissions = pd.read_csv(snakemake.input.industry_emissions).query("Fylke != 'Svalbard'")

In [ ]:
year = 2023  # 2024 and 2025 look weird with some large emitter not present (e.g. Yara)

emissions_year = (
    industry_emissions
    .loc[industry_emissions["År"] == year]
    .copy()
)


In [ ]:
emissions_year["emissions_tonnes"] = pd.to_numeric(
    emissions_year["Årlig utslipp til luft"],
    errors="coerce"
)

emissions_year = (
    emissions_year
    .dropna(subset=["emissions_tonnes"])
    .loc[lambda x: x["emissions_tonnes"] >= 0]
    .copy()
)


In [ ]:
def clean_text(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

emissions_year["Kommune_clean"] = clean_text(
    emissions_year["Kommune"]
)

emissions_year["Fylke_clean"] = clean_text(
    emissions_year["Fylke"]
)

municipalities = municipalities.copy()

municipalities["navn_clean"] = clean_text(
    municipalities["navn"]
)

municipalities["nummer"] = pd.to_numeric(
    municipalities["nummer"],
    errors="coerce"
).astype("Int64")


In [ ]:
municipality_name_number = (
    municipalities[["navn_clean", "nummer"]]
    .drop_duplicates()
    .copy()
)


In [ ]:
name_number_counts = (
    municipality_name_number
    .groupby("navn_clean")["nummer"]
    .nunique()
)

ambiguous_names = name_number_counts.loc[
    name_number_counts > 1
].index.tolist()

print("Ambiguous municipality names:")
print(ambiguous_names)


In [ ]:
municipality_lookup_unique = (
    municipality_name_number
    .loc[
        ~municipality_name_number["navn_clean"].isin(ambiguous_names)
    ]
    .rename(columns={
        "navn_clean": "Kommune_clean",
        "nummer": "nummer_name_match"
    })
    .copy()
)

assert not municipality_lookup_unique["Kommune_clean"].duplicated().any()


In [ ]:
municipality_overrides = pd.DataFrame({
    "Kommune_clean": [
        "Herøy",
        "Herøy",
        "Våler",
        "Våler i Innlandet",

    ],
    "Fylke_clean": [
        "Møre og Romsdal",
        "Nordland",
        "Østfold",
        "Innlandet",
    ],
    "nummer_override": [
        1515,  # Herøy, Møre og Romsdal
        1818,  # Herøy, Nordland
        3114,  # Våler, Østfold
        3419,  # Våler, Innlandet
    ]
})

municipality_overrides["nummer_override"] = (
    municipality_overrides["nummer_override"]
    .astype("Int64")
)


In [ ]:
emissions_year = emissions_year.merge(
    municipality_lookup_unique,
    on="Kommune_clean",
    how="left",
    validate="many_to_one"
)


In [ ]:
overrides_with_county = (
    municipality_overrides
    .loc[municipality_overrides["Fylke_clean"].notna()]
    .rename(columns={
        "nummer_override": "nummer_county_override"
    })[
        [
            "Kommune_clean",
            "Fylke_clean",
            "nummer_county_override"
        ]
    ]
)

emissions_year = emissions_year.merge(
    overrides_with_county,
    on=["Kommune_clean", "Fylke_clean"],
    how="left",
    validate="many_to_one"
)


In [ ]:
overrides_name_only = (
    municipality_overrides
    .loc[municipality_overrides["Fylke_clean"].isna()]
    .rename(columns={
        "nummer_override": "nummer_name_override"
    })[
        [
            "Kommune_clean",
            "nummer_name_override"
        ]
    ]
)

emissions_year = emissions_year.merge(
    overrides_name_only,
    on="Kommune_clean",
    how="left",
    validate="many_to_one"
)


In [ ]:
emissions_year["nummer"] = (
    emissions_year["nummer_county_override"]
    .combine_first(emissions_year["nummer_name_override"])
    .combine_first(emissions_year["nummer_name_match"])
    .astype("Int64")
)


In [ ]:
emissions_year.loc[
    emissions_year["Kommune_clean"].isin(
        ["Herøy", "Våler", "Våler i Innlandet"]
    ),
    [
        "Anleggsnavn",
        "Kommune",
        "Fylke",
        "Kommune_clean",
        "Fylke_clean",
        "nummer",
    ]
].drop_duplicates()


In [ ]:
unmatched_municipalities = (
    emissions_year.loc[
        emissions_year["nummer"].isna(),
        ["Kommune", "Fylke", "Kommune_clean", "Fylke_clean"]
    ]
    .drop_duplicates()
    .sort_values(["Fylke_clean", "Kommune_clean"])
)

print(unmatched_municipalities)


In [ ]:
total_raw_emissions = emissions_year["emissions_tonnes"].sum()

unmatched_emissions = (
    emissions_year
    .loc[emissions_year["nummer"].isna(), "emissions_tonnes"]
    .sum()
)

print(f"Total reported emissions: {total_raw_emissions:,.0f} tonnes")
print(f"Unmatched emissions:      {unmatched_emissions:,.0f} tonnes")
print(f"Unmatched share:          {unmatched_emissions / total_raw_emissions:.2%}")


In [ ]:
emissions_matched = (
    emissions_year
    .dropna(subset=["nummer"])
    .copy()
)

municipal_emissions = (
    emissions_matched
    .groupby("nummer", as_index=False)
    .agg(
        emissions_tonnes=("emissions_tonnes", "sum"),
        n_facilities=("AnleggId", "nunique")
    )
)

municipal_emissions["nummer"] = (
    municipal_emissions["nummer"]
    .astype("Int64")
)


In [ ]:
municipality_labels = (
    municipalities[["nummer", "navn"]]
    .drop_duplicates(subset="nummer")
    .copy()
)

municipal_emissions = municipal_emissions.merge(
    municipality_labels,
    on="nummer",
    how="left",
    validate="one_to_one"
)


In [ ]:
total_matched_emissions = (
    municipal_emissions["emissions_tonnes"].sum()
)

municipal_emissions["emissions_share"] = (
    municipal_emissions["emissions_tonnes"]
    / total_matched_emissions
)

municipal_emissions["new_industry_demand_twh"] = (
    municipal_emissions["emissions_share"]
    * national_new_industry_demand_twh
)

municipal_emissions["new_industry_demand_mwh"] = (
    municipal_emissions["new_industry_demand_twh"]
    * 1_000_000
)


In [ ]:
industry_allocation = municipal_emissions[
    [
        "nummer",
        "navn",
        "new_industry_demand_mwh",
        "new_industry_demand_twh",
    ]
].copy()

industry_allocation = industry_allocation.rename(columns={
    "nummer": "KOMMUNENUMMER",
    "navn": "KOMMUNE",
})

industry_allocation["KOMMUNENUMMER"] = pd.to_numeric(
    industry_allocation["KOMMUNENUMMER"],
    errors="coerce"
).astype("Int64")


In [ ]:
existing_industry_profile = (
    demand_with_all_growth
    .loc[
        demand_with_all_growth["FORBRUKSGRUPPE"].eq("Existing industry"),
        [
            "KOMMUNE",
            "KOMMUNENUMMER",
            "demand_MW",
        ]
    ]
    .copy()
    .reset_index()
)


In [ ]:
datetime_column = existing_industry_profile.columns[0]

existing_industry_profile = existing_industry_profile.rename(
    columns={datetime_column: "datetime"}
)

existing_industry_profile["KOMMUNENUMMER"] = pd.to_numeric(
    existing_industry_profile["KOMMUNENUMMER"],
    errors="coerce"
).astype("Int64")


In [ ]:
existing_industry_annual = (
    existing_industry_profile
    .groupby("KOMMUNENUMMER", as_index=False)
    .agg(
        existing_industry_mwh=("demand_MW", "sum")
    )
)


In [ ]:
industry_profile = existing_industry_profile.merge(
    existing_industry_annual,
    on="KOMMUNENUMMER",
    how="left",
    validate="many_to_one"
)

industry_profile = industry_profile.merge(
    industry_allocation[
        [
            "KOMMUNENUMMER",
            "new_industry_demand_mwh",
        ]
    ],
    on="KOMMUNENUMMER",
    how="left",
    validate="many_to_one"
)

industry_profile["new_industry_demand_mwh"] = (
    industry_profile["new_industry_demand_mwh"]
    .fillna(0)
)


In [ ]:
industry_profile["demand_MW_new_industry"] = np.where(
    industry_profile["existing_industry_mwh"] > 0,
    (
        industry_profile["demand_MW"]
        / industry_profile["existing_industry_mwh"]
        * industry_profile["new_industry_demand_mwh"]
    ),
    0
)


In [ ]:
allocated_municipalities = set(
    industry_allocation.loc[
        industry_allocation["new_industry_demand_mwh"].gt(0),
        "KOMMUNENUMMER"
    ]
    .dropna()
)

profile_municipalities = set(
    existing_industry_profile["KOMMUNENUMMER"]
    .dropna()
)

missing_profile_municipalities = (
    allocated_municipalities
    - profile_municipalities
)

print("Municipalities with allocation but no Industry profile:")
print(sorted(missing_profile_municipalities))


In [ ]:
new_industry_hourly = industry_profile[
    [
        "datetime",
        "KOMMUNE",
        "KOMMUNENUMMER",
        "demand_MW_new_industry",
    ]
].rename(columns={
    "demand_MW_new_industry": "demand_MW"
})

new_industry_hourly["FORBRUKSGRUPPE"] = (
    "Industry electrification"
)

new_industry_hourly = new_industry_hourly[
    [
        "datetime",
        "KOMMUNE",
        "KOMMUNENUMMER",
        "FORBRUKSGRUPPE",
        "demand_MW",
    ]
].set_index("datetime")


In [ ]:
national_industry_profile = (
    existing_industry_profile
    .groupby("datetime", as_index=False)
    .agg(
        national_industry_mw=("demand_MW", "sum")
    )
)

national_industry_profile["hourly_share"] = (
    national_industry_profile["national_industry_mw"]
    / national_industry_profile["national_industry_mw"].sum()
)


In [ ]:
missing_allocations = (
    industry_allocation
    .loc[
        industry_allocation["KOMMUNENUMMER"].isin(
            missing_profile_municipalities
        )
    ]
    .copy()
)


In [ ]:
fallback_hourly = (
    missing_allocations
    .merge(
        national_industry_profile[
            ["datetime", "hourly_share"]
        ],
        how="cross"
    )
)

fallback_hourly["demand_MW"] = (
    fallback_hourly["new_industry_demand_mwh"]
    * fallback_hourly["hourly_share"]
)

fallback_hourly["FORBRUKSGRUPPE"] = (
    "Industry electrification"
)

fallback_hourly = fallback_hourly[
    [
        "datetime",
        "KOMMUNE",
        "KOMMUNENUMMER",
        "FORBRUKSGRUPPE",
        "demand_MW",
    ]
].set_index("datetime")


In [ ]:
demand_with_all_growth = pd.concat(
    [
        demand_with_all_growth,
        new_industry_hourly,
        fallback_hourly,
    ],
    axis=0
).sort_index()


In [ ]:
daily_demand_by_group = (
    demand_with_all_growth
    .reset_index()
    .assign(demand_GW = lambda x : x.demand_MW/1000)
    .groupby(['datetime', 'FORBRUKSGRUPPE'])['demand_GW'].sum()
    .unstack('FORBRUKSGRUPPE')
    .resample('D').sum()
)

ax = daily_demand_by_group.plot(
    figsize=(16, 7),
    linewidth=1.2
)

ax.set_title('Daily average electricity demand by consumption group — Norway, 2050')
ax.set_xlabel('')
ax.set_ylabel('Daily demand [GWh]')
ax.legend(title='Consumption group', ncol=4)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
demand_with_all_growth.demand_MW.sum()/1000/1000

In [ ]:
demand_with_all_growth.sort_values(by='KOMMUNENUMMER')

## Data centres

In [ ]:
# https://www.statnett.no/nettkapasitet-til-produksjon-og-forbruk/foresporsler-og-reservasjon-i-nettet/#reservasjoner
# We use the reserved capacity in the grid and scale it with respect to that
# dc_built = pd.read_csv('../../resources/cap_built_data_centres.csv') # These are already included in the data - should only be the reserved capacity.
dc_reserved = pd.read_csv(snakemake.input.datacentres)
national_new_datacentre_demand_twh = 18
national_new_datacentre_demand_mwh = (
    national_new_datacentre_demand_twh * 1_000_000
)

In [ ]:
# There is an extra column "Dato" that should not be in the data. This below cleans that without modifying the raw input data from Elhub.
dc_reserved.columns = (
    dc_reserved.columns
    .str.strip()
)

dc_reserved = dc_reserved.rename(columns={
    "Dato": "Planlagt tilknytning",
    "Planlagt tiknytning": "Kunde og tilknytningsansvarlig_raw",
    "Kunde og tilknytningsansvarlig": "unused_empty_column",
})

dc_reserved = dc_reserved.drop(
    columns="unused_empty_column",
    errors="ignore"
)


In [ ]:
capacity_col = "Reservert kapasitet (MW)"
station_col = "Stasjon for tilknytning i transmisjonsnettet"
customer_col = "Sluttkunde"

dc_reserved["reserved_capacity_mw"] = pd.to_numeric(
    dc_reserved[capacity_col],
    errors="coerce"
)

dc_reserved = dc_reserved.dropna(
    subset=["reserved_capacity_mw", station_col]
).copy()

dc_reserved = dc_reserved.loc[
    dc_reserved["reserved_capacity_mw"] > 0
].copy()

dc_reserved["reserved_capacity_mw"] = pd.to_numeric(
    dc_reserved["Reservert kapasitet (MW)"],
    errors="coerce",
)

dc_reserved = dc_reserved.rename(columns={
    "Kunde og tilknytningsansvarlig_raw":
        "Kunde og tilknytningsansvarlig"
})


In [ ]:
dc_reserved["planned_connection_date"] = pd.to_datetime(
    dc_reserved["Planlagt tilknytning"],
    dayfirst=True,
    errors="coerce"
)

In [ ]:
reserved_mw = dc_reserved["reserved_capacity_mw"].sum()  # 4,217 MW

implied_cf = (
    national_new_datacentre_demand_twh * 1_000_000
    / (reserved_mw * 8760)
)

average_load_mw = (
    national_new_datacentre_demand_twh * 1_000_000
    / 8760
)

print(implied_cf)      # ~0.487
print(average_load_mw) # ~2,055 MW

# A historic capacity factor would result in ~ 0.43
# 3.3*1000*1000/(dc_built['Tilknyttet kapasitet totalt (MW)'].sum()*8760)

In [ ]:
# Needs to  be updated if new projects in new municipalities are included in the data sheet. 

sluttkunde_to_municipality = {
    'Tegneby TRA' : 'Vestby',
    'Kristiansand TRA' : 'Vennesla',
    'Fagrafjell TRA' : 'Sandnes', # Could also be Time as it is on border...
    'Ertsmyra TS TRA' : 'Sirdal',
    'Ringerike TRA' : 'Ringerike',
    'Kvandal TRA' : 'Narvik',
    'Brokke KRA/TRA' : 'Valle',
    'Rød TRA' : 'Skien',
    'Leirdøla (LEX) TRA' : 'Luster',
    'Spanne TRA' : 'Karmøy',
    'Bjerkreim TRA' : 'Bjerkreim',
    'Tunnsjødal KRA/TRA' : 'Namsskogan',
    'Vang TRA' : 'Hamar',
    'Ålfoten TRA': 'Bremanger',
    'Grenland TRA' : 'Skien',
    'Kobbvatnet TRA' : 'Sørfold',
    'Frogner TRA' : 'Lillestrøm',
    'Rana KRA/TRA' : 'Rana',
    'Vinje KRA' : 'Vinje',
    'Rjukan TRA' : 'Tinn',
    'Haugsvær transformatorstasjon TRA' : 'Masfjorden',
    'Svartisen KRA/TRA' : 'Meløy',
    'Ulven TRA' : 'Oslo',
    'Marka TRA' : 'Vefsn',
    'Fræna TRA' : 'Hustadvika',
    'Porsgrunn TRA' : 'Porsgrunn',
    'Salten transformatorstasjon TRA' : 'Sørfold',
    'Stølaheia TRA' : 'Stavanger',
    'Vågåmo TRA' : 'Vågå',
    'Bærheim TRA' : 'Sandnes',
    'Sautso TRA' : 'Alta',
    'Hinnøy' : 'Sortland',
    'Åfjord TRA' : 'Åfjord',
}

In [ ]:
dc_reserved['municipality'] = dc_reserved[station_col].map(
    sluttkunde_to_municipality
)

In [ ]:
unmatched_stations = (
    dc_reserved.loc[
        dc_reserved['municipality'].isna(),
        station_col
    ]
    .dropna()
    .unique()
)

print(unmatched_stations)


In [ ]:
dc_capacity_by_municipality = (
    dc_reserved
    .dropna(subset=["municipality"])
    .groupby("municipality", as_index=False)["reserved_capacity_mw"]
    .sum()
    .rename(columns={"reserved_capacity_mw": "reserved_dc_capacity_mw"})
)

dc_capacity_by_municipality["demand_MW"] = (
    dc_capacity_by_municipality["reserved_dc_capacity_mw"]
    * implied_cf
)

dc_capacity_by_municipality


In [ ]:
# All timestamps in the master DataFrame
hours = demand_with_all_growth.index.unique()

# Municipality -> municipality number lookup
municipality_numbers = (
    demand_with_all_growth[
        ["KOMMUNE", "KOMMUNENUMMER"]
    ]
    .drop_duplicates(subset="KOMMUNE")
    .set_index("KOMMUNE")["KOMMUNENUMMER"]
)

# Cartesian product: every hour x every municipality with a data centre
dc_hourly_load = (
    pd.MultiIndex.from_product(
        [hours, dc_capacity_by_municipality["municipality"]],
        names=["datetime", "KOMMUNE"],
    )
    .to_frame(index=False)
    .merge(
        dc_capacity_by_municipality[
            ["municipality", "reserved_dc_capacity_mw", "demand_MW"]
        ],
        left_on="KOMMUNE",
        right_on="municipality",
        how="left",
    )
)

dc_hourly_load["KOMMUNENUMMER"] = dc_hourly_load["KOMMUNE"].map(
    municipality_numbers
)

dc_hourly_load["FORBRUKSGRUPPE"] = "New data centres"

# Retain only the same schema as the master DataFrame
dc_hourly_load = (
    dc_hourly_load[
        [
            "datetime",
            "KOMMUNE",
            "KOMMUNENUMMER",
            "FORBRUKSGRUPPE",
            "demand_MW",
        ]
    ]
    .set_index("datetime")
    .sort_index()
)


In [ ]:
dc_annual_energy_mwh = (
    dc_hourly_load["demand_MW"].sum()
)

dc_annual_energy_twh = dc_annual_energy_mwh / 1_000_000

print(f"Data-centre annual demand: {dc_annual_energy_twh:.3f} TWh")
print(
    f"Target annual demand:      "
    f"{national_new_datacentre_demand_twh:.3f} TWh"
)


In [ ]:
demand_with_all_growth = (
    pd.concat(
        [demand_with_all_growth, dc_hourly_load],
        axis=0,
    )
    .sort_index()
)

In [ ]:
demand_with_all_growth.sort_values(by='KOMMUNENUMMER')

## Hydrogen

In [ ]:
NATIONAL_H2_TWH = 27.0
N_MUNICIPALITIES = len(municipalities)

CAPACITY_FACTOR = 0.70
ELECTROLYSER_EFFICIENCY = 0.70

In [ ]:
hydrogen_statnett = pd.read_csv(snakemake.input.hydrogen)

In [ ]:
capacity_col = "Kapasitet i kø (MW)"
hydrogen_statnett["reserved_capacity_mw"] = pd.to_numeric(
    hydrogen_statnett[capacity_col],
    errors="coerce"
)

In [ ]:
hydrogen_statnett.reserved_capacity_mw.sum()

In [ ]:
pattern = "|".join(re.escape(str(key)) for key in sluttkunde_to_municipality.keys())
col = "Stasjon for tilknytning i transmisjonsnettet"
unmatched = (
    hydrogen_statnett.loc[
        ~hydrogen_statnett[col].fillna("").astype(str).str.contains(pattern, case=False, na=False),
        col
    ]
    .dropna()
    .unique()
    .tolist()
)

In [ ]:
# This requires some manual digging. 
sluttkunde_to_municipality.update({
    'Tveiten TRA' : 'Tønsberg',
    'Sildvik KRA' : 'Narvik', 
    'Ørskog TRA' : 'Ålesund',
    'Moskog TRA' : 'Sunnfjord',
    'Varangerbotn TRA' : 'Unjárga', # Nesseby
    'Viklandet TRA' : 'Sunndal',
    'Nedre Røssåga KRA/TRA' : 'Hemnes',
    'Kjelland TRA' : 'Eigersund',
    'Håvik TRA' : 'Karmøy',
    'Lindås transformatorstasjon TRA' : 'Alver',
    'Skaidi TRA' : 'Hammerfest',
    'Mestervik' : 'Balsfjord',
    'Eidum TRA' : 'Stjørdal',
    'Kollsnes KRA/TRA' : 'Øygarden',
    'Høyanger TRA' : 'Høyanger',
})


In [ ]:
station_col = "Stasjon for tilknytning i transmisjonsnettet"

In [ ]:
hydrogen_statnett['municipality'] = hydrogen_statnett[station_col].map(
    sluttkunde_to_municipality
)

In [ ]:
unmatched_stations = (
    hydrogen_statnett.loc[
        hydrogen_statnett['municipality'].isna(),
        station_col
    ]
    .dropna()
    .unique()
)

print(unmatched_stations)


In [ ]:
hydrogen_capacity_by_municipality = (
    hydrogen_statnett
    .dropna(subset=["municipality"])
    .groupby("municipality", as_index=False)["reserved_capacity_mw"]
    .sum()
    .rename(columns={"reserved_capacity_mw": "reserved_hydrogen_capacity_mw"})
)

hydrogen_capacity_by_municipality["demand_MW"] = (
    hydrogen_capacity_by_municipality["reserved_hydrogen_capacity_mw"]
    * CAPACITY_FACTOR
)

In [ ]:
hydrogen_capacity_by_municipality

In [ ]:
# All timestamps in the master DataFrame
hours = demand_with_all_growth.index.unique()

# Municipality -> municipality number lookup
municipality_numbers = (
    demand_with_all_growth[
        ["KOMMUNE", "KOMMUNENUMMER"]
    ]
    .drop_duplicates(subset="KOMMUNE")
    .set_index("KOMMUNE")["KOMMUNENUMMER"]
)

# Cartesian product: every hour x every municipality with a data centre
hydrogen_hourly_load = (
    pd.MultiIndex.from_product(
        [hours, hydrogen_capacity_by_municipality["municipality"]],
        names=["datetime", "KOMMUNE"],
    )
    .to_frame(index=False)
    .merge(
        hydrogen_capacity_by_municipality[
            ["municipality", "reserved_hydrogen_capacity_mw", "demand_MW"]
        ],
        left_on="KOMMUNE",
        right_on="municipality",
        how="left",
    )
)

hydrogen_hourly_load["KOMMUNENUMMER"] = hydrogen_hourly_load["KOMMUNE"].map(
    municipality_numbers
)

hydrogen_hourly_load["FORBRUKSGRUPPE"] = "New hydrogen production"

# Retain only the same schema as the master DataFrame
hydrogen_hourly_load = (
    hydrogen_hourly_load[
        [
            "datetime",
            "KOMMUNE",
            "KOMMUNENUMMER",
            "FORBRUKSGRUPPE",
            "demand_MW",
        ]
    ]
    .set_index("datetime")
    .sort_index()
)


In [ ]:
hydrogen_annual_energy_mwh = (
    hydrogen_hourly_load["demand_MW"].sum()
)

hydrogen_annual_energy_twh = hydrogen_annual_energy_mwh / 1_000_000

In [ ]:
# Avoid negative remainder if the project estimate exceeds national demand
remainder_twh = max(0, NATIONAL_H2_TWH - hydrogen_annual_energy_twh)
equal_remainder_per_municipality = remainder_twh / N_MUNICIPALITIES

print(f"Estimated existing-project output: {hydrogen_annual_energy_twh:.3f} TWh/year")
print(f"National scenario:                 {NATIONAL_H2_TWH:.3f} TWh/year")
print(f"Remainder for equal allocation:    {remainder_twh:.3f} TWh/year")
print(
    "Equal allocation per municipality: "
    f"{equal_remainder_per_municipality * 1000:.1f} GWh/year"
)

In [ ]:
# Use the municipality universe from the master demand dataset
municipality_lookup = (
    demand_with_all_growth[
        ["KOMMUNE", "KOMMUNENUMMER"]
    ]
    .dropna(subset=["KOMMUNE", "KOMMUNENUMMER"])
    .drop_duplicates(subset=["KOMMUNE"])
    .sort_values("KOMMUNE")
    .reset_index(drop=True)
)

N_MUNICIPALITIES = len(municipality_lookup)

# Calculate the actual number of represented hours.
# This is safer than assuming 8,760 if you later use another year/time resolution.
n_hours = len(hours)

# TWh/year -> MWh/year, then divide evenly over municipalities and hours.
remainder_demand_mw_per_municipality = (
    remainder_twh * 1_000_000 / N_MUNICIPALITIES / n_hours
)

print(
    f"Flat remainder load per municipality: "
    f"{remainder_demand_mw_per_municipality:.3f} MW"
)


In [ ]:
hydrogen_remainder_hourly_load = (
    pd.MultiIndex.from_product(
        [
            hours,
            municipality_lookup["KOMMUNE"],
        ],
        names=["datetime", "KOMMUNE"],
    )
    .to_frame(index=False)
    .merge(
        municipality_lookup,
        on="KOMMUNE",
        how="left",
        validate="many_to_one",
    )
    .assign(
        FORBRUKSGRUPPE="New hydrogen production",
        demand_MW=remainder_demand_mw_per_municipality,
    )
    [
        [
            "datetime",
            "KOMMUNE",
            "KOMMUNENUMMER",
            "FORBRUKSGRUPPE",
            "demand_MW",
        ]
    ]
    .set_index("datetime")
    .sort_index()
)


In [ ]:
hydrogen_total_hourly_load = (
    pd.concat(
        [
            hydrogen_hourly_load,
            hydrogen_remainder_hourly_load,
        ],
        axis=0,
    )
    .sort_index()
)


In [ ]:
hydrogen_total_hourly_load = (
    hydrogen_total_hourly_load
    .reset_index()
    .groupby(
        [
            "datetime",
            "KOMMUNE",
            "KOMMUNENUMMER",
            "FORBRUKSGRUPPE",
        ],
        as_index=False,
        dropna=False,
    )["demand_MW"]
    .sum()
    .set_index("datetime")
    .sort_index()
)


In [ ]:
hydrogen_total_annual_twh = (
    hydrogen_total_hourly_load["demand_MW"].sum() / 1_000_000
)

print(f"Allocated hydrogen demand: {hydrogen_total_annual_twh:.3f} TWh/year")

assert np.isclose(
    hydrogen_total_annual_twh,
    NATIONAL_H2_TWH,
    rtol=1e-8,
), "Hydrogen allocation does not sum to the national target."


In [ ]:
demand_with_all_growth = (
    pd.concat(
        [demand_with_all_growth, hydrogen_total_hourly_load],
        axis=0,
    )
    .sort_index()
)

In [ ]:
daily_demand_by_group = (
    demand_with_all_growth
    .reset_index()
    .assign(demand_GW = lambda x : x.demand_MW/1000)
    .groupby(['datetime', 'FORBRUKSGRUPPE'])['demand_GW'].sum()
    .unstack('FORBRUKSGRUPPE')
    .resample('D').sum()
)

ax = daily_demand_by_group.plot(
    figsize=(16, 7),
    linewidth=1.2
)

ax.set_title('Daily average electricity demand by consumption group — Norway, 2050')
ax.set_xlabel('')
ax.set_ylabel('Daily demand [GWh]')
ax.legend(title='Consumption group', ncol=4)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
demand_with_all_growth.demand_MW.sum()/1000/1000

In [ ]:
demand_with_all_growth.sort_values(by='KOMMUNENUMMER')

## Transportation

In [ ]:
"""
Data on existing vehicles is available from SSB
The zonal distribution of the EVs is based on the current number of registered private cars in each county in Norway in 2025
Retain the share of cars in each municipality, but absolute volume increase to 3,247,639
"""

In [ ]:
national_new_transport_demand_twh = 22
ev_consumption = 0.19 #kWh/km

In [ ]:
# Municipal-level car fleet registry https://www.ssb.no/statbank/table/11823/
municipal_car_fleet = pd.read_csv(snakemake.input.car_fleet,sep=';',encoding='latin',skiprows=1)

# Municipal-level road volumne https://www.ssb.no/en/statbank/table/12579
municipal_car_volume = pd.read_csv(snakemake.input.car_volume,sep=';',encoding='latin',skiprows=1)

In [ ]:
municipal_car_fleet

In [ ]:
municipal_car_fleet_elec = municipal_car_fleet.query("drivstofftype == 'El.'")

In [ ]:
municipal_car_fleet_elec = (
    municipal_car_fleet_elec
    .groupby('region').sum()
    .assign(
        number_elec_cars = lambda x : x['Personbiler 2025 Euro-0'] + x['Personbiler 2025 Euro-1'] + x['Personbiler 2025 Euro-2'] + x['Personbiler 2025 Euro-3'] + x['Personbiler 2025 Euro-4'] + x['Personbiler 2025 Euro-5'] + x['Personbiler 2025 Euro-6'] + x['Personbiler 2025 Uoppgitt'],
    )
    .loc[:,['number_elec_cars']]
    .reset_index()
    .assign(
        municipality_id=lambda x: x['region'].str.extract(r'^K-(\d{4})').astype('Int64'),
        municipality_name=lambda x: (
            x['region']
            .str.replace(r'^K-\d{4}\s+', '', regex=True)
            .str.split(r'\s+-\s+', n=1)
            .str[0]
        )
    )
    .drop(columns='region')
)

municipal_car_fleet_elec = municipal_car_fleet_elec.loc[municipal_car_fleet_elec['municipality_id'].notna()]


In [ ]:
municipal_car_volume = (
    municipal_car_volume
    .loc[:,['region','Average road traffic volumes per vehicle (km) 2025 All passenger cars']]
    .assign(
        region_code = lambda x : x.region.str.split(' ').str[0],
        region_name = lambda x : x.region.str.split(' ').str[1],
    )
    .loc[:,['region_name','region_code','Average road traffic volumes per vehicle (km) 2025 All passenger cars']]
)

In [ ]:
municipal_car_volume['region_code'] = municipal_car_volume['region_code'].astype('Int64')

In [ ]:
municipal_car_fleet = (
    municipal_car_fleet
    .groupby('region').sum()
    .assign(
        number_cars = lambda x : x['Personbiler 2025 Euro-0'] + x['Personbiler 2025 Euro-1'] + x['Personbiler 2025 Euro-2'] + x['Personbiler 2025 Euro-3'] + x['Personbiler 2025 Euro-4'] + x['Personbiler 2025 Euro-5'] + x['Personbiler 2025 Euro-6'] + x['Personbiler 2025 Uoppgitt'],
        share_cars = lambda x : x.number_cars/x.number_cars.sum()
    )
    .loc[:,['number_cars','share_cars']]
    .reset_index()
    .assign(
        municipality_id=lambda x: x['region'].str.extract(r'^K-(\d{4})').astype('Int64'),
        municipality_name=lambda x: (
            x['region']
            .str.replace(r'^K-\d{4}\s+', '', regex=True)
            .str.split(r'\s+-\s+', n=1)
            .str[0]
        )
    )
    .drop(columns='region')
)

In [ ]:
municipal_car_fleet = municipal_car_fleet.loc[municipal_car_fleet['municipality_id'].notna()]

In [ ]:
municipal_car_volume = municipal_car_volume[municipal_car_volume.region_code != 9999]

In [ ]:
municipal_data = (
    municipal_car_fleet
    .merge(
        municipal_car_volume[
            [
                'region_code',
                'Average road traffic volumes per vehicle (km) 2025 All passenger cars'
            ]
        ],
        how='left',
        left_on='municipality_id',
        right_on='region_code',
        validate='one_to_one'
    )
    .drop(columns='region_code')
    .rename(columns={
        'Average road traffic volumes per vehicle (km) 2025 All passenger cars':
            'avg_road_traffic_km_2025'
    })
)


In [ ]:
municipal_data = municipal_data.merge(
    municipal_car_fleet_elec[
        ['municipality_id', 'number_elec_cars']
    ],
    how='left',
    on='municipality_id',
    validate='one_to_one'
)

In [ ]:
municipal_data.sort_values(by='municipality_id')

In [ ]:
existing_evs = municipal_data.number_elec_cars.sum()

In [ ]:
existing_cars = municipal_data.number_cars.sum()

In [ ]:
evs_2050 = 3247639 # From Hvidsten et al. 

In [ ]:
new_evs = evs_2050-existing_evs

In [ ]:
new_evs

In [ ]:
municipal_data = (
    municipal_data
    .assign(
        normalized_share=lambda x: x['share_cars'] / x['share_cars'].sum(),
        tot_evs_2050=lambda x: (x['share_cars'] / x['share_cars'].sum()) * evs_2050,
        new_evs = lambda x : x.tot_evs_2050 - x.number_elec_cars
    )
)


In [ ]:
municipal_data['new_evs'] = municipal_data['new_evs'].round().astype(int)

difference = evs_2050 - (
    municipal_data['number_elec_cars'].sum()
    + municipal_data['new_evs'].sum()
)

municipal_data.loc[municipal_data.index[-1], 'new_evs'] += difference


In [ ]:
print(f"Allocated total EVs in 2050: {municipal_data.number_elec_cars.sum() + municipal_data.new_evs.sum()}")
print(f"Target total EVs in 2050: {evs_2050}")

In [ ]:
municipal_data = (
    municipal_data
    .assign(
        share_new_ev = lambda x : x.new_evs/x.new_evs.sum(),
        new_ev_demand_twh = lambda x : x.share_new_ev*national_new_transport_demand_twh,
        demand_MW = lambda x : (x.new_ev_demand_twh/8760)*1e6
    )
)

In [ ]:
municipal_data.sort_values(by='municipality_id')

In [ ]:
demand_with_all_growth.sort_values(by='KOMMUNENUMMER')

In [ ]:
# THIS IS CURRENTLY WITH A FLAT LOAD CURVE WHICH OF COURSE IS NOT THE CASE
# All timestamps in the master DataFrame
hours = demand_with_all_growth.index.unique()

# Municipality -> municipality number lookup
municipality_numbers = (
    demand_with_all_growth[
        ["KOMMUNE", "KOMMUNENUMMER"]
    ]
    .drop_duplicates(subset="KOMMUNE")
    .set_index("KOMMUNE")["KOMMUNENUMMER"]
)

# Cartesian product: every hour x every municipality with a data centre
ev_hourly_load = (
    pd.MultiIndex.from_product(
        [hours, municipal_data['municipality_id']],
        names=["datetime", "KOMMUNENUMMER"],
    )
    .to_frame(index=False)
    .merge(
        municipal_data[
            ["municipality_id", "demand_MW"]
        ],
        left_on="KOMMUNENUMMER",
        right_on="municipality_id",
        how="left",
    )
)

In [ ]:
ev_hourly_load["KOMMUNE"] = ev_hourly_load["KOMMUNENUMMER"].map(
    municipality_names_map
)

ev_hourly_load["FORBRUKSGRUPPE"] = "New EV demand"

# Retain only the same schema as the master DataFrame
ev_hourly_load = (
    ev_hourly_load[
        [
            "datetime",
            "KOMMUNE",
            "KOMMUNENUMMER",
            "FORBRUKSGRUPPE",
            "demand_MW",
        ]
    ]
    .set_index("datetime")
    .sort_index()
)


In [ ]:
ev_hourly_load.sort_values(by='KOMMUNENUMMER')

In [ ]:
ev_total_annual_twh = (
    ev_hourly_load["demand_MW"].sum() / 1_000_000
)

print(f"Allocated EV demand: {ev_total_annual_twh:.3f} TWh/year")

In [ ]:
demand_with_all_growth = (
    pd.concat(
        [demand_with_all_growth, ev_hourly_load],
        axis=0,
    )
    .sort_index()
)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7),linewidth=1.2)

daily_demand_by_group = (
    demand_with_all_growth
    .reset_index()
    .assign(demand_GW = lambda x : x.demand_MW/1000)
    .groupby(['datetime', 'FORBRUKSGRUPPE'])['demand_GW'].sum()
    .unstack('FORBRUKSGRUPPE')
    .resample('D').sum()
)

daily_demand_by_group.plot(ax=ax)

ax.set_title('Daily average electricity demand by consumption group — Norway, 2050')
ax.set_xlabel('')
ax.set_ylabel('Daily demand [GWh]')
ax.legend(title='Consumption group', ncol=4)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/diagnostics/electricity_demand_by_sector_norway.pdf',bbox_inches='tight')

In [ ]:
import matplotlib as mpl
# Optional: use serif fonts similar to a typical LaTeX document
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "legend.fontsize": 8,
    "legend.title_fontsize": 8,
    "pdf.fonttype": 42,  # Embed TrueType fonts in PDF
    "ps.fonttype": 42,
})

daily_energy_by_group = (
    demand_with_all_growth
    .reset_index()
    .assign(demand_GW=lambda x: x["demand_MW"] / 1000)
    .groupby(["datetime", "FORBRUKSGRUPPE"])["demand_GW"]
    .sum()
    .unstack("FORBRUKSGRUPPE")
    .resample("D")
    .sum()
)

# Roughly full text width in a standard LaTeX document
fig, ax = plt.subplots(figsize=(6.5, 3.2), constrained_layout=True)

daily_energy_by_group.plot(
    ax=ax,
    linewidth=1.0,
    alpha=0.9,
)

ax.set_xlabel("")
ax.set_ylabel("Daily electricity demand [GWh]")

ax.grid(
    axis="y",
    color="0.85",
    linewidth=0.6,
)
ax.grid(axis="x", visible=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Put the legend outside / below the plot
ax.legend(
    title="Consumption group",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=3,
    frameon=False,
)

fig.savefig(
    "figures/diagnostics/electricity_demand_by_sector_norway.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)

In [ ]:
demand_with_all_growth.sort_values(by='KOMMUNENUMMER')

## Spatial allocation

In [ ]:
municipality_names = (
    municipalities[["nummer", "navn"]]
    .dropna(subset=["nummer"])
    .drop_duplicates(subset="nummer")
    .copy()
)

In [ ]:
municipalities_unique = (
    municipalities[["nummer", "geometry"]]
    .dropna(subset=["nummer"])
    .dissolve(by="nummer", as_index=False)
    .merge(
        municipality_names,
        on="nummer",
        how="left",
        validate="one_to_one"
    )
)

municipalities_unique = gpd.GeoDataFrame(
    municipalities_unique,
    geometry="geometry",
    crs=municipalities.crs
)


In [ ]:
demand_for_maps = demand_with_all_growth.copy()

demand_for_maps["KOMMUNENUMMER"] = pd.to_numeric(
    demand_for_maps["KOMMUNENUMMER"],
    errors="coerce"
).astype("Int64")

annual_by_group = (
    demand_for_maps
    .groupby(
        ["KOMMUNENUMMER", "FORBRUKSGRUPPE"],
        as_index=False
    )
    .agg(
        annual_demand_mwh=("demand_MW", "sum"),
        peak_demand_mw=("demand_MW", "max")
    )
)

annual_by_group["annual_demand_twh"] = (
    annual_by_group["annual_demand_mwh"] / 1_000_000
)

annual_by_group.head()


In [ ]:
# groups = sorted(
#     annual_by_group["FORBRUKSGRUPPE"]
#     .dropna()
#     .unique()
# )
# 
# ncols = 3
# nrows = int(np.ceil(len(groups) / ncols))
# 
# fig, axes = plt.subplots(
#     nrows=nrows,
#     ncols=ncols,
#     figsize=(18, 6 * nrows)
# )
# 
# axes = np.asarray(axes).flatten()
# 
# for ax, group in zip(axes, groups):
# 
#     group_demand = annual_by_group.loc[
#         annual_by_group["FORBRUKSGRUPPE"].eq(group),
#         ["KOMMUNENUMMER", "annual_demand_twh"]
#     ]
# 
#     group_map = municipalities_unique.merge(
#         group_demand,
#         left_on="nummer",
#         right_on="KOMMUNENUMMER",
#         how="left",
#         validate="one_to_one"
#     )
# 
#     group_map["annual_demand_twh"] = (
#         group_map["annual_demand_twh"]
#         .fillna(0)
#     )
# 
#     zero_map = group_map.loc[
#         group_map["annual_demand_twh"] <= 0
#     ]
# 
#     positive_map = group_map.loc[
#         group_map["annual_demand_twh"] > 0
#     ]
# 
#     zero_map.plot(
#         ax=ax,
#         color="lightgrey",
#         edgecolor="white",
#         linewidth=0.1
#     )
# 
#     if not positive_map.empty:
#         positive_map.plot(
#             column="annual_demand_twh",
#             cmap="Blues",
#             norm=LogNorm(
#                 vmin=positive_map["annual_demand_twh"].min(),
#                 vmax=positive_map["annual_demand_twh"].max()
#             ),
#             legend=True,
#             edgecolor="white",
#             linewidth=0.1,
#             ax=ax,
#             legend_kwds={
#                 "label": "TWh/year (log)",
#                 "shrink": 0.5
#             }
#         )
# 
#     ax.set_title(group, fontsize=12)
#     ax.set_axis_off()
# 
# for ax in axes[len(groups):]:
#     ax.set_visible(False)
# 
# fig.suptitle(
#     "Annual electricity demand by municipality and consumption group",
#     fontsize=18,
#     y=0.98
# )
# 
# plt.tight_layout()
# plt.savefig('figures/diagnostics/annual_municipal_elec_demand_by_sector.pdf',bbox_inches='tight')
# plt.show()


In [ ]:
annual_total = (
    annual_by_group
    .groupby("KOMMUNENUMMER", as_index=False)
    .agg(
        annual_demand_mwh=("annual_demand_mwh", "sum")
    )
)

annual_total["annual_demand_twh"] = (
    annual_total["annual_demand_mwh"] / 1_000_000
)


In [ ]:
total_map = municipalities_unique.merge(
    annual_total[
        [
            "KOMMUNENUMMER",
            "annual_demand_mwh",
            "annual_demand_twh"
        ]
    ],
    left_on="nummer",
    right_on="KOMMUNENUMMER",
    how="left",
    validate="one_to_one"
)

total_map["annual_demand_twh"] = (
    total_map["annual_demand_twh"]
    .fillna(0)
)


In [ ]:
#fig, ax = plt.subplots(figsize=(10, 14))
#
#vmin, vmax = np.nanquantile(
#    total_map.annual_demand_twh.values,
#    [0.02, 0.98],
#)
#
#total_map.plot(
#    column="annual_demand_twh",
#    cmap="Blues",
#    norm=LogNorm(vmin=vmin, vmax=vmax),
#    #scheme="quantiles",
#    #k=6,
#    legend=True,
#    edgecolor="white",
#    linewidth=0.15,
#    ax=ax,
#    legend_kwds={
#        "label": "Annual electricity demand (TWh)",
#        "shrink": 0.6
#    }
#)
#
#ax.set_title(
#    "Total annual electricity demand by municipality\nAll consumption groups",
#    fontsize=14
#)
#ax.set_axis_off()
#
#plt.tight_layout()
#plt.savefig('figures/diagnostics/municipal_elec_demand_map.pdf',bbox_inches='tight')
#plt.show()


In [ ]:
#daily_demand_all = (
#    demand_with_all_growth
#    .reset_index()
#    .assign(demand_GW = lambda x : x.demand_MW/1000)
#    .groupby(['datetime'])['demand_GW'].sum()
#    #.unstack('FORBRUKSGRUPPE')
#    .resample('D').sum()
#)
#
#daily_demand_orig = (
#    demand
#    .reset_index()
#    .assign(demand_GW = lambda x : x.demand_MW/1000)
#    .groupby(['datetime'])['demand_GW'].sum()
#    #.unstack('FORBRUKSGRUPPE')
#    .resample('D').sum()
#)
#
#fig, ax = plt.subplots()
#
#daily_demand_all.plot(
#    ax=ax,
#    figsize=(16, 7),
#    linewidth=1.2,
#    label='Future demand'
#)
#
#daily_demand_orig.plot(
#    ax=ax,
#    figsize=(16, 7),
#    linewidth=1.2,
#    label='Current demand'
#)
#
#ax.set_title('Daily average electricity demand - Norway, 2050')
#ax.set_xlabel('')
#ax.set_ylabel('Daily demand [GWh]')
#ax.set_ylim(0,daily_demand_all.max()*1.1)
#ax.legend(title='Consumption group', ncol=4)
#ax.grid(True, alpha=0.3)
#
#plt.tight_layout()
#plt.savefig('figures/diagnostics/daily_demand_currentVSfuture.pdf',bbox_inches='tight')
#plt.show()

In [ ]:
demand_with_all_growth.sort_values(by='KOMMUNENUMMER')

## Mapping to spatial resolution

In [ ]:
shape = gpd.read_file(snakemake.input.shapefile)
norway = shape.query("zone.str.contains('NO')")
norway = gpd.GeoDataFrame(
    norway,
    geometry="geometry",
    crs=norway.crs if hasattr(norway, "crs") else "EPSG:4326",
)

In [ ]:
municipalities_unique = gpd.GeoDataFrame(
    municipalities_unique,
    geometry="geometry",
    crs=municipalities_unique.crs if hasattr(municipalities_unique, "crs") else "EPSG:25833",
)

municipalities_unique = municipalities_unique.to_crs(norway.crs)

In [ ]:
muni_points = municipalities_unique[["nummer", "navn", "geometry"]].copy()

muni_points["geometry"] = muni_points.geometry.representative_point()

muni_points = gpd.GeoDataFrame(
    muni_points,
    geometry="geometry",
    crs=municipalities_unique.crs,
)


In [ ]:
norway_zones = norway[["zone", "geometry"]].copy()

municipality_zone = gpd.sjoin(
    muni_points,
    norway_zones,
    how="left",
    predicate="within",
)[["nummer", "navn", "zone"]].copy()


In [ ]:
municipality_zone[municipality_zone["zone"].isna()]

In [ ]:
municipality_zone["nummer"] = municipality_zone["nummer"].astype("int64")
demand_with_all_growth["KOMMUNENUMMER"] = demand_with_all_growth["KOMMUNENUMMER"].astype("int64")

In [ ]:
unmatched = municipality_zone[municipality_zone["zone"].isna()]
print(unmatched[["nummer", "navn"]])

In [ ]:
duplicates = municipality_zone[
    municipality_zone.duplicated("nummer", keep=False)
].sort_values("nummer")

print(duplicates)


In [ ]:
demand_with_all_growth

In [ ]:
demand_tmp = demand_with_all_growth.copy()
demand_tmp.index.name = "datetime"
demand_tmp = demand_tmp.reset_index()

In [ ]:
municipality_hourly = (
    demand_tmp
    .groupby(
        ["datetime", "KOMMUNENUMMER"],
        observed=True,
        sort=False,
        as_index=False,
    )["demand_MW"]
    .sum()
)


In [ ]:
zone_hourly = (
    municipality_hourly
    .merge(
        municipality_zone[["nummer", "zone"]],
        left_on="KOMMUNENUMMER",
        right_on="nummer",
        how="left",
        validate="many_to_one",
    )
    .groupby(["datetime", "zone"], observed=True, as_index=False)["demand_MW"]
    .sum()
)

In [ ]:
#fig, ax = plt.subplots(figsize=(16, 7), linewidth=1.2)
#
#daily_demand_by_zone = (
#    zone_hourly
#    .assign(demand_GW=lambda x: x["demand_MW"] / 1000)
#    .set_index("datetime")
#    .groupby("zone")["demand_GW"]
#    .resample("D")
#    .sum()
#    .unstack("zone")
#)
#
#daily_demand_by_zone.plot(ax=ax)
#
#ax.set_title("Daily electricity demand by zone — Norway, 2050")
#ax.set_xlabel("")
#ax.set_ylabel("Daily demand [GWh]")
#ax.legend(title="Zone", ncol=4)
#ax.grid(True, alpha=0.3)
#
#plt.tight_layout()
#plt.savefig('figures/diagnostics/daily_elec_demand_byZone.pdf',bbox_inches='tight')


In [ ]:
demand_tmp = (
    demand_with_all_growth
    .rename_axis("datetime")
    .reset_index()
)

# Ensure municipality IDs use the same dtype on both sides
demand_tmp["KOMMUNENUMMER"] = demand_tmp["KOMMUNENUMMER"].astype(int)
municipality_zone["nummer"] = municipality_zone["nummer"].astype(int)

demand_with_zone = demand_tmp.merge(
    municipality_zone[["nummer", "zone"]],
    left_on="KOMMUNENUMMER",
    right_on="nummer",
    how="left",
    validate="many_to_one",
)


In [ ]:
annual_by_zone_group = (
    demand_with_zone
    .dropna(subset=["zone"])
    .groupby(
        ["zone", "FORBRUKSGRUPPE"],
        observed=True,
        as_index=False,
    )["demand_MW"]
    .sum()
    .assign(
        annual_demand_twh=lambda x: x["demand_MW"] / 1_000_000
    )
)

annual_by_zone_group.head()


In [ ]:
annual_by_zone_group.FORBRUKSGRUPPE.unique()

In [ ]:
round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['Existing commercial'].annual_demand_twh,1)

In [ ]:
title_numbers = {
    'Existing commercial' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['Existing commercial'].annual_demand_twh,1),
    'Existing industry' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['Existing industry'].annual_demand_twh,1),
    'Existing residential' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['Existing residential'].annual_demand_twh,1),
    'Industry electrification' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['Industry electrification'].annual_demand_twh,1),
    'New EV demand' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['New EV demand'].annual_demand_twh,1),
    'New data centres' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['New data centres'].annual_demand_twh,1),
    'New hydrogen production' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['New hydrogen production'].annual_demand_twh,1),
    'Petroleum electrification' : round(annual_by_zone_group.groupby('FORBRUKSGRUPPE').sum().loc['Petroleum electrification'].annual_demand_twh,1),
}

In [ ]:
#annual_by_zone_group_plot = annual_by_zone_group.loc[
#    annual_by_zone_group["FORBRUKSGRUPPE"].ne("General change")
#].copy()
#
#groups = sorted(
#    annual_by_zone_group_plot["FORBRUKSGRUPPE"]
#    .dropna()
#    .unique()
#)
#
## Fewer columns = much larger maps per panel
#ncols = 2
#nrows = int(np.ceil(len(groups) / ncols))
#
#fig, axes = plt.subplots(
#    nrows=nrows,
#    ncols=ncols,
#    figsize=(16, 8 * nrows),  # Large landscape layout
#    constrained_layout=True,  # Better than tight_layout with legends/colorbars
#)
#
#axes = np.atleast_1d(axes).flatten()
#
#for ax, group in zip(axes, groups):
#
#    group_demand = annual_by_zone_group_plot.loc[
#        annual_by_zone_group_plot["FORBRUKSGRUPPE"].eq(group),
#        ["zone", "annual_demand_twh"],
#    ]
#
#    group_map = norway.merge(
#        group_demand,
#        on="zone",
#        how="left",
#        validate="one_to_one",
#    )
#
#    group_map["annual_demand_twh"] = (
#        group_map["annual_demand_twh"].fillna(0)
#    )
#
#    # Background: zones without demand
#    group_map.plot(
#        ax=ax,
#        color="lightgrey",
#        edgecolor="white",
#        linewidth=0.7,
#    )
#
#    positive_map = group_map.loc[
#        group_map["annual_demand_twh"].gt(0)
#    ]
#
#    if not positive_map.empty:
#        vmin = positive_map["annual_demand_twh"].min()
#        vmax = positive_map["annual_demand_twh"].max()
#
#        if vmin == vmax:
#            positive_map.plot(
#                ax=ax,
#                color=plt.cm.Blues(0.7),
#                edgecolor="white",
#                linewidth=0.7,
#            )
#        else:
#            positive_map.plot(
#                ax=ax,
#                column="annual_demand_twh",
#                cmap="Blues",
#                norm=LogNorm(vmin=vmin, vmax=vmax),
#                legend=True,
#                edgecolor="white",
#                linewidth=0.7,
#                legend_kwds={
#                    "label": "Annual demand [TWh]",
#                    "shrink": 0.70,
#                    "pad": 0.02,
#                },
#            )
#
#    total_twh = title_numbers.get(group)
#
#    ax.set_title(
#        f"{group} ({total_twh:.1f} TWh)",
#        fontsize=16,
#        fontweight="bold",
#        pad=10,
#    )
#    ax.set_axis_off()
#
## Hide spare panels
#for ax in axes[len(groups):]:
#    ax.set_visible(False)
#
#fig.suptitle(
#    "Annual electricity demand by zone and consumption group",
#    fontsize=22,
#    fontweight="bold",
#)
#
#fig.savefig(
#    "figures/diagnostics/annual_demand_by_zone_sector.pdf",
#    format="pdf",
#    bbox_inches="tight",
#)
#
#plt.show()


## Prepare output

In [ ]:
zone_hourly_export = zone_hourly.copy()

zone_hourly_export["datetime"] = (
    zone_hourly_export["datetime"]
    .dt.tz_localize(None)
)

zone_hourly_wide = (
    zone_hourly_export
    .pivot(
        index="datetime",
        columns="zone",
        values="demand_MW",
    )
    .sort_index()
    .reset_index()
)

zone_hourly_wide.columns.name = None

In [ ]:
zone_hourly_wide["datetime"] = zone_hourly_wide["datetime"].map(lambda x : x.replace(year=2010))

In [ ]:
zone_hourly_wide

In [ ]:
zone_hourly_wide.to_csv(
    snakemake.output.norway_hourly_demand_MW,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S",
)